In [1]:
import pandas as pd
import numpy as np

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from sentence_transformers import CrossEncoder

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
import sys
sys.path.append("..")

In [3]:
# Load Data

books = pd.read_csv("../data/books_cleaned.csv")

In [4]:
# Creates documents with metadata

documents = []

for _, row in books.iterrows():

    doc = Document(
        page_content=row["tagged_description"],

        metadata={
            "isbn13": int(row["isbn13"]),
            "title": row.get("title", ""),
            "authors": row.get("authors", ""),
            "categories": row.get("categories", ""),
            "published_year": row.get("published_year", "")
        }
    )

    documents.append(doc)

In [5]:
# Embedding Model

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"batch_size": 64}
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
# Vector database

db_books = Chroma.from_documents(
    documents,
    embedding=embedding_model,
    persist_directory="../vector_db/"
)

In [7]:
# Keyword Search (TF-IDF)

tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(books["tagged_description"])

In [8]:
# Reranking Model

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
# Keyword search function

def keyword_search(query, top_k=20):

    query_vec = tfidf.transform([query])

    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

    top_indices = scores.argsort()[-top_k:][::-1]

    return books.iloc[top_indices]

In [10]:
# Hybrid Recommendation Function

def retrieve_semantic_recommendations(
        query: str,
        top_k: int = 10,
        category_filter: str | None = None,
        user_preferences: list | None = None
):

    # Semantic Search

    semantic_results = db_books.similarity_search(query, k=30)

    semantic_isbns = [doc.metadata["isbn13"] for doc in semantic_results]

    # Keyword Search

    keyword_results = keyword_search(query, top_k=30)

    keyword_isbns = keyword_results["isbn13"].tolist()

    # Hybrid Candidate Pool

    candidate_isbns = list(set(semantic_isbns + keyword_isbns))

    candidates = books[books["isbn13"].isin(candidate_isbns)]

    # Category Filtering

    if category_filter:

        candidates = candidates[
            candidates["categories"]
            .str.contains(category_filter, case=False, na=False)
        ]

    if len(candidates) == 0:
        return pd.DataFrame()

    # Reranking

    pairs = [(query, text) for text in candidates["tagged_description"]]

    rerank_scores = reranker.predict(pairs)

    candidates = candidates.copy()

    candidates["rerank_score"] = rerank_scores

    # Popularity / Rating Signals

    candidates["rating_score"] = candidates["average_rating"].fillna(0)

    candidates["popularity_score"] = np.log1p(
        candidates["ratings_count"].fillna(0)
    )


    # Final ranking score

    candidates["final_score"] = (
        0.6 * candidates["rerank_score"]
        + 0.3 * candidates["rating_score"]
        + 0.1 * candidates["popularity_score"]
    )


    # Personalization
    if user_preferences:

        preference_bonus = candidates["categories"].apply(
            lambda x: any(pref.lower() in str(x).lower() for pref in user_preferences)
        )

        candidates.loc[preference_bonus, "final_score"] += 0.5

    # Final Sorting

    candidates = candidates.sort_values(
        "final_score",
        ascending=False
    )


    return candidates.head(top_k)


In [11]:
print(
    retrieve_semantic_recommendations(
        "A book to teach children about nature"
    )
)

             isbn13      isbn10                                         title  \
2457  9780465016907  0465016901                 The Drama of the Gifted Child   
4264  9780941807555  094180755X        The Little Big Book for God's Children   
324   9780060959036  0060959037                               Prodigal Summer   
3747  9780786808069  0786808063           Baby Einstein: Neighborhood Animals   
3000  9780671631987  0671631985  Teach Your Child to Read in 100 Easy Lessons   
1078  9780241003008  0241003008                   The Very Hungry Caterpillar   
1108  9780285647473  0285647474                 Zen Buddhism & Psychoanalysis   
3749  9780786808380  0786808381                         Baby Einstein: Babies   
3185  9780688140717  0688140718                           Trapped in T Mirror   
3760  9780786812912  0786812915                                   The Big Box   

                                                authors  \
2457                                             

In [12]:
print(
    retrieve_semantic_recommendations(
        "Books about space exploration",
        category_filter="science"
    )
)

             isbn13      isbn10                       title  \
727   9780141011110  0141011114    The Fabric of the Cosmos   
2707  9780553380163  0553380168     A Brief History of Time   
1922  9780393324464  039332446X     The Future of Spacetime   
2481  9780471409762  0471409766               Bad Astronomy   
5018  9781852339463  1852339462  The Future of the Universe   
1439  9780345251220  0345251229        Visions from Nowhere   
2790  9780553802023  055380202X  The Universe in a Nutshell   

                                                authors       categories  \
727                                        Brian Greene          Science   
2707                                    Stephen Hawking          Science   
1922  Stephen W. Hawking;Kip S. Thorne;Igor D. Novik...          Science   
2481                                    Philip C. Plait          Science   
5018                                       A.J. Meadows          Science   
1439                                   

In [13]:
print(
    retrieve_semantic_recommendations(
        "fantasy adventure magic",
        user_preferences=["fantasy"]
    )
)

             isbn13      isbn10                        title  \
2945  9780618894642  0618894640            Children of Húrin   
2225  9780441005963  0441005969                Riddle-master   
4455  9781405201711  1405201711     The Faraway Tree Stories   
2634  9780552548229  0552548227                Spindle's End   
5046  9781857237504  1857237501       The Conan Chronicles 1   
3976  9780812575606  0812575601           The Sword of Truth   
2875  9780595092147  0595092144                    Firedrake   
3536  9780755311514  0755311515  The Science of Harry Potter   
3818  9780802135780  0802135781            Sexing the Cherry   
2855  9780586071397  0586071393                  Faerie Tale   

                        authors                    categories  \
2945  John Ronald Reuel Tolkien                       Fiction   
2225       Patricia A. McKillip                       Fiction   
4455                Enid Blyton   Children's stories, English   
2634             Robin McKinley    